## create schema


In [0]:
%sql
create schema if not exists workspace.bronze

In [0]:
from pyspark.sql.functions import current_timestamp, col

volume_path = "/Volumes/workspace/default/raw_data"
target_schema = "workspace.bronze"

raw_tables = {
    "bronze_aisles": f"{volume_path}/aisles.csv",
    "bronze_departments": f"{volume_path}/departments.csv",
    "bronze_products": f"{volume_path}/products.csv",
    "bronze_orders": f"{volume_path}/orders.csv",
    "bronze_order_products_prior": f"{volume_path}/order_products__prior.csv",
    "bronze_order_products_train": f"{volume_path}/order_products__train.csv"
}

for table_name, file_path in raw_tables.items():
    print(f"Writing Delta table: {target_schema}.{table_name}...")
    
    (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(file_path)
        .withColumn("_ingest_timestamp", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{target_schema}.{table_name}")
    )

print("All Bronze tables successfully created in Delta format!")

In [0]:
%sql
show tables in workspace.bronze